# 02 Preprocessing and Feature Engineering

After loading and inspecting the dataset, this notebook preprocesses the SCANIA Component X operational readouts, time-to-event labels, and vehicle specifications for the subsequent tabular and sequence survival models considered in the study. Here, sequences will be built at the vehicle level, followed by splits (train, val, test) and feature representations to create a survival dataset for non-sequential models.

The main purpose is to construct tabular vehicle-level features from the raw operational histories and prepare the corresponding survival targets.

## Inputs

- `data/raw/train_operational_readouts.csv`
- `data/raw/train_specifications.csv`
- `data/raw/train_tte.csv`

## Outputs
- processed sequence files
- processed tabular feature files
- processed survival target files

In [51]:
# Set project paths and enable imports from the src directory

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

print(f"Notebook location: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")

Notebook location: c:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\notebooks
Project root: c:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x
Source directory: c:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\src


In [52]:
# Load the data
from config import OPERATIONAL_FILE, SPECIFICATIONS_FILE, TTE_FILE
from data_loading import load_raw_training_data
operational_df, spec_df, tte_df = load_raw_training_data(
    OPERATIONAL_FILE,
    SPECIFICATIONS_FILE,
    TTE_FILE
)

Operational loaded successfully: (1122452, 107)
Specifications loaded successfully: (23550, 9)
TTE loaded successfully: (23550, 3)


In [53]:
# Sort operational readouts by vehicle_id and time_step
from quality_checks import sort_operational_data

operational_sorted_df = sort_operational_data(operational_df)

Operational data sorted by vehicle_id and time_step.
Sorted shape: (1122452, 107)


In [54]:
# Split histogram features from numerical features
from preprocessing import split_operational_feature_groups

feature_groups = split_operational_feature_groups(operational_df)

id_columns = feature_groups["id_columns"]
numerical_feature_cols = feature_groups["numerical_feature_cols"]
histogram_feature_cols = feature_groups["histogram_feature_cols"]
histogram_groups = feature_groups["histogram_groups"]
# check if the operation was successful
print("Numerical features:", numerical_feature_cols)
print("Histogram group keys:", list(histogram_groups.keys()))
# for histogram features with more than 5 bins, print the first 5 bins
for group_name, cols in histogram_groups.items():
    print(group_name, len(cols), cols[:5], "..." if len(cols) > 5 else "")

Identifier/time columns: ['vehicle_id', 'time_step']
Numerical scalar feature count: 8
Histogram feature count: 97
Histogram group count: 6
Numerical features: ['100_0', '171_0', '309_0', '370_0', '427_0', '666_0', '835_0', '837_0']
Histogram group keys: ['167', '272', '291', '158', '459', '397']
167 10 ['167_0', '167_1', '167_2', '167_3', '167_4'] ...
272 10 ['272_0', '272_1', '272_2', '272_3', '272_4'] ...
291 11 ['291_0', '291_1', '291_2', '291_3', '291_4'] ...
158 10 ['158_0', '158_1', '158_2', '158_3', '158_4'] ...
459 20 ['459_0', '459_1', '459_2', '459_3', '459_4'] ...
397 36 ['397_0', '397_1', '397_2', '397_3', '397_4'] ...


In [55]:
# Building operational sequences at vehicle level
from sequence_builder import build_operational_sequences
# dictionary of sequences, each vehicle_id is also a dictionary itself
operational_sequences = build_operational_sequences(
    operational_sorted_df,
    numerical_feature_cols,
    histogram_feature_cols
)

Built operational sequences.
Vehicle sequence count: 23550


In [56]:
# Check on the built sequences, retrieve info of one vehicle
vehicle_5_seq = operational_sequences[5]

print("Vehicle ID:", vehicle_5_seq["vehicle_id"])
print("Sequence length:", vehicle_5_seq["sequence_length"])
print("Time steps shape:", vehicle_5_seq["time_steps"].shape)
print("Time gaps shape:", vehicle_5_seq["time_gaps"].shape)
print("Numerical sequence shape:", vehicle_5_seq["numerical_sequence"].shape)
print("Histogram sequence shape:", vehicle_5_seq["histogram_sequence"].shape)

print("\nFirst 5 time steps:")
print(vehicle_5_seq["time_steps"][:5])

print("\nFirst 5 time gaps:")
print(vehicle_5_seq["time_gaps"][:5])

Vehicle ID: 5
Sequence length: 44
Time steps shape: (44,)
Time gaps shape: (44,)
Numerical sequence shape: (44, 8)
Histogram sequence shape: (44, 97)

First 5 time steps:
[ 0.   0.2  1.2  1.6 16.6]

First 5 time gaps:
[ 0.   0.2  1.   0.4 15. ]


In [57]:
# check sequence length information
from sequence_builder import summarize_sequence_lengths, sequence_length_table
sequence_length_summary = summarize_sequence_lengths(operational_sequences)

Sequence-length summary:
count    23550.000000
mean        47.662505
std         27.406028
min          5.000000
25%         28.000000
50%         43.000000
75%         64.000000
90%         83.000000
95%         97.000000
99%        129.510000
max        303.000000
Name: sequence_length, dtype: float64


In [58]:
# Create a table with sequence length information of each vehicle
sequence_length_df = sequence_length_table(operational_sequences)
sequence_length_df.head()

Sequence-length table shape: (23550, 2)


,vehicle_id,sequence_length
0,0,172
1,2,33
2,3,71
3,4,17
4,5,44


In [59]:
# Sort the vehicles by sequence length to see which has shortest and longest sequence
sequence_length_df.sort_values("sequence_length", ascending=True).head()


,vehicle_id,sequence_length
7175,10286,5
23506,33584,5
8007,11468,5
2000,2840,5
20580,29441,5


In [60]:
sequence_length_df.sort_values("sequence_length", ascending=False).head()

,vehicle_id,sequence_length
1948,2762,303
211,293,237
162,228,236
239,329,231
1330,1888,230


In [61]:
# create another with static features
# first merge time to event with vehicle specification
from data_loading import merge_vehicle_level_data

vehicle_df = merge_vehicle_level_data(spec_df, tte_df)
vehicle_df.head()

Merged vehicle-level dataframe shape: (23550, 11)


,vehicle_id,Spec_0,Spec_1,Spec_2,Spec_3,Spec_4,Spec_5,Spec_6,Spec_7,length_of_study_time_step,in_study_repair
0,0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0
1,2,Cat0,Cat1,Cat1,Cat0,Cat0,Cat0,Cat0,Cat1,281.8,0
2,3,Cat0,Cat1,Cat1,Cat1,Cat0,Cat0,Cat0,Cat1,293.4,0
3,4,Cat0,Cat0,Cat2,Cat1,Cat0,Cat0,Cat0,Cat1,210.0,0
4,5,Cat0,Cat2,Cat2,Cat0,Cat0,Cat0,Cat0,Cat1,360.4,0


In [62]:
from sequence_builder import attach_vehicle_level_data_to_sequences
enriched_sequences_with_static = attach_vehicle_level_data_to_sequences(
    operational_sequences,
    vehicle_df
)

Sequences enriched with vehicle-level data: 23550
Vehicle IDs missing from vehicle_df: 0


In [63]:
# check the resulting dictionary
vehicle_0_enriched_with_static = enriched_sequences_with_static[0]

print("Vehicle ID:", vehicle_0_enriched_with_static["vehicle_id"])
print("Sequence length:", vehicle_0_enriched_with_static["sequence_length"])
print("Duration:", vehicle_0_enriched_with_static["duration"])
print("Event:", vehicle_0_enriched_with_static["event"])
print("Static features:")
print(vehicle_0_enriched_with_static["static_features"])

Vehicle ID: 0
Sequence length: 172
Duration: 510.0
Event: 0
Static features:
{'Spec_0': 'Cat0', 'Spec_1': 'Cat0', 'Spec_2': 'Cat0', 'Spec_3': 'Cat0', 'Spec_4': 'Cat0', 'Spec_5': 'Cat0', 'Spec_6': 'Cat0', 'Spec_7': 'Cat0'}


## Train Test Split
### 1. Full history
Before doing any preprocessing and feature selection actions which may transform the data and introduce leakage risks, the dataset is split into train, validation and test subsets. All actions will be fitted on the training set only and applied to the validation and test sets.

In [64]:
# To avoid leakage, split the sequence dictionary first
from sklearn.model_selection import train_test_split
vehicle_ids = list(enriched_sequences_with_static.keys())
events = [enriched_sequences_with_static[veh_id]["event"] for veh_id in vehicle_ids]
train_ids, temp_ids = train_test_split(
    vehicle_ids, 
    test_size = 0.30,
    random_state = 42, 
    stratify = events
)

temp_events = [enriched_sequences_with_static[veh_id]["event"] for veh_id in temp_ids]
val_ids, test_ids = train_test_split(
    temp_ids,
    test_size = 0.50,
    random_state = 42,
    stratify = temp_events
)

In [65]:
# rebuild sequences dictionaries
train_seq = {veh_id: enriched_sequences_with_static[veh_id] for veh_id in train_ids}
val_seq = {veh_id: enriched_sequences_with_static[veh_id] for veh_id in val_ids}
test_seq = {veh_id: enriched_sequences_with_static[veh_id] for veh_id in test_ids}

print(len(train_seq), len(val_seq), len(test_seq))

16485 3532 3533


In [66]:
train_seq[18034]["numerical_sequence"]

array([[1.07920600e+06, 3.15090000e+05, 0.00000000e+00, 0.00000000e+00,
        1.26947920e+07, 2.63200000e+03, 1.30322810e+07, 4.00000000e+01],
       [2.70035100e+06, 7.81485000e+05, 0.00000000e+00, 0.00000000e+00,
        3.14505510e+07, 5.04000000e+03, 3.23413910e+07, 5.60000000e+01],
       [3.01550600e+06, 8.74185000e+05, 0.00000000e+00, 0.00000000e+00,
        3.52462770e+07, 5.73300000e+03, 3.62244010e+07, 5.60000000e+01],
       [3.35012200e+06, 9.48345000e+05, 0.00000000e+00, 0.00000000e+00,
        3.82722010e+07, 6.23000000e+03, 3.94755310e+07, 7.20000000e+01],
       [4.51396700e+06, 1.26508500e+06, 0.00000000e+00, 0.00000000e+00,
        5.09798520e+07, 9.83500000e+03, 5.26803110e+07, 1.13000000e+02],
       [6.36020200e+06, 1.84339500e+06, 0.00000000e+00, 0.00000000e+00,
        7.39718100e+07, 1.52460000e+04, 7.60716510e+07, 1.78000000e+02],
       [7.76044300e+06, 2.24323500e+06, 0.00000000e+00, 0.00000000e+00,
        9.00440540e+07, 1.92640000e+04, 9.26351410e+07, 3.

### 2. Truncated sequences

To reflect practical predictive maintenance settings, construct a prediction-time version of the full sequences by truncating each vehicle trajectory at a randomly sampled readout. For each truncated trajectory, redefine the target duration as the remaining time-to-event from the sampled readout. The resulting truncated histories are then used as input for both tabular aggregation and sequence modeling. To ensure independence in randomness, each split has its own seed for this truncation process.

In [67]:
# Create truncated versions after splitting
import importlib
import sequence_builder as sequence_builder
importlib.reload(sequence_builder)
train_seq_truncated = sequence_builder.apply_random_readout_to_sequences(
    enriched_sequences=train_seq,
    min_history_points=5,
    random_state=42
)

val_seq_truncated = sequence_builder.apply_random_readout_to_sequences(
    enriched_sequences=val_seq,
    min_history_points=5,
    random_state=43
)

test_seq_truncated = sequence_builder.apply_random_readout_to_sequences(
    enriched_sequences=test_seq,
    min_history_points=5,
    random_state=44
)

print(len(train_seq_truncated), len(val_seq_truncated), len(test_seq_truncated))

Kept 16485 truncated sequences.
Skipped 0 sequences.
Kept 3532 truncated sequences.
Skipped 0 sequences.
Kept 3533 truncated sequences.
Skipped 0 sequences.
16485 3532 3533


In [23]:
# save sequences in their split and trajectory information differences
# first full histories
import pickle
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
SEQUENCE_DIR = PROJECT_ROOT / "data" / "processed" / "sequences"
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

# Save full-history split dictionaries
with open(SEQUENCE_DIR / "train_seq_full.pkl", "wb") as f:
    pickle.dump(train_seq, f)

with open(SEQUENCE_DIR / "val_seq_full.pkl", "wb") as f:
    pickle.dump(val_seq, f)

with open(SEQUENCE_DIR / "test_seq_full.pkl", "wb") as f:
    pickle.dump(test_seq, f)

print("Saved full-history split sequence dictionaries.")

Saved full-history split sequence dictionaries.


In [38]:
# then truncated histories

with open(SEQUENCE_DIR / "train_seq_truncated.pkl", "wb") as f:
    pickle.dump(train_seq_truncated, f)

with open(SEQUENCE_DIR / "val_seq_tuncated.pkl", "wb") as f:
    pickle.dump(val_seq_truncated, f)

with open(SEQUENCE_DIR / "test_seq_truncated.pkl", "wb") as f:
    pickle.dump(test_seq_truncated, f)

print("Saved truncated histories split sequence dictionaries.")

Saved truncated histories split sequence dictionaries.


## Survival dataset creation for tabular survival models
Manually aggregated features that reflect degradation-related information from summary of multivariate time series are provided to give the tabular models input that preserve temporal structure of the data. To reiterate, information provided consist of operational histories available up to the randomly selected readout point. This is to reflect practical predictive maintenance setting, where reliability must be estimated when the vehicle is still operational.

In [68]:
# Now aggregate for tabular baselines
from feature_engineering import aggregate_sequences_for_tabular_baselines
X_train_trunc, y_train_trunc = aggregate_sequences_for_tabular_baselines(
    train_seq_truncated,
    numerical_feature_cols,
    histogram_feature_cols
)

X_val_trunc, y_val_trunc = aggregate_sequences_for_tabular_baselines(
    val_seq_truncated,
    numerical_feature_cols,
    histogram_feature_cols
)

X_test_trunc, y_test_trunc = aggregate_sequences_for_tabular_baselines(
    test_seq_truncated,
    numerical_feature_cols,
    histogram_feature_cols
)

Aggregated predictors dataframe shape: (16485, 275)
Aggregated targets dataframe shape: (16485, 2)
Vehicle count: 16485
Aggregated predictors dataframe shape: (3532, 275)
Aggregated targets dataframe shape: (3532, 2)
Vehicle count: 3532
Aggregated predictors dataframe shape: (3533, 275)
Aggregated targets dataframe shape: (3533, 2)
Vehicle count: 3533


## Preprocessing stage
In this stage, two datasets will be created for an ablation study on whether to keep or discard categorical variables. This is motivated by prior study on this dataset, which found out that these vehicle specifications do not add predictive information to modelling. This might be easier as they constitute a large part of the dimensionality. Including them must be beneficial to all models. Here I start with building a dataset that include them. Preprocessing actions such as one hot encode and missing value imputation, as well as feature selection actions of removing low variance and highly correlated features will include them in the first dataset. Then another will be formed excluding them in both processing and feature selection. 

In [69]:
# create only dynamic features dataset, dropping static features
static_cols = [col for col in X_train_trunc.columns if col.startswith("Spec_")]

X_train_trunc_no_static = X_train_trunc.drop(columns=static_cols, errors="ignore").copy()
X_val_trunc_no_static = X_val_trunc.drop(columns=static_cols, errors="ignore").copy()
X_test_trunc_no_static = X_test_trunc.drop(columns=static_cols, errors="ignore").copy()

In [70]:
# drop vehicle_id, not used during modelling
X_train_trunc_no_static = X_train_trunc_no_static.drop(columns=["vehicle_id"], errors= "ignore").copy()
X_val_trunc_no_static = X_val_trunc_no_static.drop(columns=["vehicle_id"], errors= "ignore").copy()
X_test_trunc_no_static = X_test_trunc_no_static.drop(columns=["vehicle_id"], errors= "ignore").copy()

In [71]:
# Impute missing values, fit on truncated training set
from preprocessing import fit_and_apply_imputer_train
X_train_trunc_ns_imputed, tabular_trunc_imputer_ns = fit_and_apply_imputer_train(
    X_train_trunc_no_static
)

Imputed predictor matrix shape: (16485, 266)
Remaining missing values: 0


In [72]:
# Apply on truncated validation and test sets
from preprocessing import apply_tabular_imputer
X_val_trunc_ns_imputed = apply_tabular_imputer(
    X_val_trunc_no_static,
    tabular_trunc_imputer_ns
)
X_test_trunc_ns_imputed = apply_tabular_imputer(
    X_test_trunc_no_static,
    tabular_trunc_imputer_ns
)

Applied imputer to predictor matrix shape: (3532, 266)
Remaining missing values: 0
Applied imputer to predictor matrix shape: (3533, 266)
Remaining missing values: 0


### One hot encode static features 
For the dataset with both dynamic and static features, categorical features must be encoded. Fit encoder on the training set and apply on the val and test sets


In [73]:
from preprocessing import fit_and_apply_one_hot_train, apply_one_hot_to_other_split
X_train_trunc_ws, encoded_columns = fit_and_apply_one_hot_train(X_train_trunc, static_cols)
X_val_trunc_ws = apply_one_hot_to_other_split(
    X=X_val_trunc, 
    categorical_cols= static_cols, 
    train_columns=encoded_columns)
X_test_trunc_ws = apply_one_hot_to_other_split(
    X= X_test_trunc,
    categorical_cols= static_cols,
    train_columns=encoded_columns
)

In [74]:
# remove vehicle id, not used in modelling
X_train_trunc_with_static = X_train_trunc_ws.drop(columns=["vehicle_id"], errors= "ignore").copy()
X_val_trunc_with_static = X_val_trunc_ws.drop(columns=["vehicle_id"], errors= "ignore").copy()
X_test_trunc_with_static = X_test_trunc_ws.drop(columns=["vehicle_id"], errors= "ignore").copy()

## Feature selection workflow
For the ablation analysis, feature selection is conducted separately for each dataset. The dataset of dynamic features only might have features that correlate with each other without influence of categorical variables. 

In [75]:
# remove low-info (li) variables, apply function on training set only
from preprocessing import remove_near_zero_variance_features, remove_highly_correlated_features
X_train_trunc_ns_rli, removed_li_trunc_ns_vars, li_selector_ns = remove_near_zero_variance_features(
    X_train_trunc_ns_imputed, threshold = 1e-4
)
X_val_trunc_ns_rli = X_val_trunc_ns_imputed.drop(columns=removed_li_trunc_ns_vars, errors= "ignore")
X_test_trunc_ns_rli = X_test_trunc_ns_imputed.drop(columns=removed_li_trunc_ns_vars, errors= "ignore")

# remove highly correlated features, apply function on train set only
X_train_trunc_ns_final, removed_corr_trunc_ns = remove_highly_correlated_features(X_train_trunc_ns_rli, threshold=0.90)
X_val_trunc_ns_final = X_val_trunc_ns_rli.drop(columns=removed_corr_trunc_ns, errors="ignore")
X_test_trunc_ns_final = X_test_trunc_ns_rli.drop(columns=removed_corr_trunc_ns, errors="ignore")

Original shape: (16485, 266)
Reduced shape after near-zero variance filtering: (16485, 266)
Removed feature count: 0
Original shape: (16485, 266)
Reduced shape after correlation filtering: (16485, 92)
Removed correlated feature count: 174


In [76]:
# Check the first 40 features removed
print("Removed correlated features:", len(removed_corr_trunc_ns))
print(removed_corr_trunc_ns[:40])
print("Final training predictor shape:", X_train_trunc_ns_final.shape)

Removed correlated features: 174
['100_0_delta', '100_0_max', '100_0_mean', '100_0_min', '100_0_std', '158_0_mean', '158_1_last', '158_2_last', '158_2_mean', '158_3_mean', '158_4_mean', '158_5_last', '158_5_mean', '158_6_mean', '158_7_last', '158_7_mean', '158_8_mean', '158_9_mean', '167_0_mean', '167_1_last', '167_1_mean', '167_2_mean', '167_3_mean', '167_4_mean', '167_5_last', '167_5_mean', '167_6_mean', '167_7_mean', '167_8_mean', '167_9_mean', '171_0_delta', '171_0_first', '171_0_last', '171_0_max', '171_0_mean', '171_0_min', '171_0_slope', '171_0_std', '272_0_mean', '272_1_last']
Final training predictor shape: (16485, 92)


In [77]:
# check kept features
kept_cols_ns = X_train_trunc_ns_final.columns.tolist()

numerical_kept_ns = [col for col in kept_cols_ns if any(col.startswith(f"{f}_") for f in numerical_feature_cols)]
histogram_kept_ns = [col for col in kept_cols_ns if any(col.startswith(f"{h}_") for h in histogram_feature_cols)]
gap_kept_ns = [col for col in kept_cols_ns if col.startswith("gap_")]
metadata_kept_ns = [col for col in kept_cols_ns if col in ["sequence_length", "final_time_step", "observation_density"]]

print("Kept numerical summary features:", len(numerical_kept_ns))
print("Kept histogram summary features:", len(histogram_kept_ns))
print("Kept gap features:", len(gap_kept_ns))
print("Kept metadata features:", len(metadata_kept_ns))


Kept numerical summary features: 18
Kept histogram summary features: 67
Kept gap features: 4
Kept metadata features: 3


In [78]:
# print the first 40 kept numerical features
print(numerical_kept_ns[:40])

['100_0_first', '100_0_last', '100_0_slope', '309_0_last', '309_0_delta', '309_0_slope', '370_0_first', '370_0_last', '370_0_slope', '427_0_slope', '666_0_last', '666_0_slope', '835_0_first', '835_0_last', '835_0_slope', '837_0_last', '837_0_delta', '837_0_slope']


In [79]:
# save the preprocessing history of the dataset in a dictionary
metadata_ns_final = {
    "low_info_removed": removed_li_trunc_ns_vars,
    "corr_removed": removed_corr_trunc_ns,
    "final_columns": X_train_trunc_ns_final.columns.tolist(),
    "imputer": tabular_trunc_imputer_ns
}

In [47]:
# save the datasets without static features

TABULAR_DIR = PROJECT_ROOT / "data" / "processed" / "tabular_truncated"
TABULAR_DIR.mkdir(parents=True, exist_ok=True)

with open(TABULAR_DIR / "tabular_trunc_without_static_final.pkl", "wb") as f:
    pickle.dump(
        {
            "X_train_trunc_ns": X_train_trunc_ns_final,
            "X_val_trunc_ns": X_val_trunc_ns_final,
            "X_test_trunc-ns": X_test_trunc_ns_final,
            "y_train_trunc": y_train_trunc,
            "y_val_trunc": y_val_trunc,
            "y_test_trunc": y_test_trunc,
            "metadata_trunc": metadata_kept_ns,
        },
        f
    )

#### Preprocessing and feature selection for dataset with both dynamic and static features

In [42]:
# Impute missing values, fit on training set

X_train_trunc_ws_imputed, tabular_imputer_ws = fit_and_apply_imputer_train(
    X_train_trunc_with_static
)

Imputed predictor matrix shape: (16485, 347)
Remaining missing values: 0


In [41]:
# Then apply on validation and test sets
X_val_trunc_ws_imputed = apply_tabular_imputer(
    X_val_trunc_with_static,
    tabular_imputer_ws
)
X_test_trunc_ws_imputed = apply_tabular_imputer(
    X_test_trunc_with_static,
    tabular_imputer_ws
)

Applied imputer to predictor matrix shape: (3532, 347)
Remaining missing values: 0
Applied imputer to predictor matrix shape: (3533, 347)
Remaining missing values: 0


In [44]:
# remove low-info (li) variables, apply function on truncated training set only

X_train_trunc_ws_rli, removed_li_trunc_ws_vars, li_selector_trunc_ws = remove_near_zero_variance_features(
    X_train_trunc_ws_imputed, threshold = 1e-4
)
X_val_trunc_ws_rli = X_val_trunc_ws_imputed.drop(columns=removed_li_trunc_ws_vars, errors= "ignore")
X_test_trunc_ws_rli = X_test_trunc_ws_imputed.drop(columns=removed_li_trunc_ws_vars, errors= "ignore")

# remove highly correlated features, apply function on train set only
X_train_trunc_ws_final, removed_corr_trunc_ws = remove_highly_correlated_features(X_train_trunc_ws_rli, threshold=0.90)
X_val_trunc_ws_final = X_val_trunc_ws_rli.drop(columns=removed_corr_trunc_ws, errors="ignore")
X_test_trunc_ws_final = X_test_trunc_ws_rli.drop(columns=removed_corr_trunc_ws, errors="ignore")

Original shape: (16485, 347)
Reduced shape after near-zero variance filtering: (16485, 345)
Removed feature count: 2
Original shape: (16485, 345)
Reduced shape after correlation filtering: (16485, 169)
Removed correlated feature count: 176


In [45]:
# save the preprocessing history of the truncated dataset in a dictionary
metadata_trunc_ws_final = {
    "categorical_cols": static_cols,
    "train_encoded_columns": X_train_trunc_with_static.columns.tolist(),
    "low_info_removed": removed_li_trunc_ws_vars,
    "corr_removed": removed_corr_trunc_ws,
    "final_columns": X_train_trunc_ws_final.columns.tolist(),
    "imputer": tabular_imputer_ws
}

In [46]:
# save csv files
from utils import save_dataframe_csv, save_list_json, save_dict_json

### Save both datasets


In [48]:

save_dataframe_csv(
    X_train_trunc_ns_final,
    TABULAR_DIR / "X_train_without_static.csv"
)

save_dataframe_csv(
    X_val_trunc_ns_final,
    TABULAR_DIR / "X_val_without_static.csv"
)

save_dataframe_csv(
    X_test_trunc_ns_final,
    TABULAR_DIR / "X_test_without_static.csv"
)


Saved dataframe to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\X_train_without_static.csv
Saved dataframe to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\X_val_without_static.csv
Saved dataframe to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\X_test_without_static.csv


In [49]:
save_dataframe_csv(
    X_train_trunc_ws_final,
    TABULAR_DIR / "X_train_trunc_with_static.csv"
)

save_dataframe_csv(
    X_val_trunc_ws_final,
    TABULAR_DIR / "X_val_trunc_with_static.csv"
)

save_dataframe_csv(
    X_test_trunc_ws_final,
    TABULAR_DIR / "X_test_trunc_with_static.csv"
)


Saved dataframe to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\X_train_trunc_with_static.csv
Saved dataframe to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\X_val_trunc_with_static.csv
Saved dataframe to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\X_test_trunc_with_static.csv


In [80]:
# Save targets dataset
save_dataframe_csv(
    y_train_trunc,
    TABULAR_DIR / "y_train_trunc.csv"
)
save_dataframe_csv(
    y_val_trunc,
    TABULAR_DIR / "y_val_trunc.csv"
)
save_dataframe_csv(
    y_test_trunc,
    TABULAR_DIR / "y_test_trunc.csv"
)

Saved dataframe to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\y_train_trunc.csv
Saved dataframe to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\y_val_trunc.csv
Saved dataframe to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\y_test_trunc.csv


In [50]:
# save useful metadata
save_list_json(
    X_train_trunc_with_static.columns.tolist(),
    TABULAR_DIR / "static_encoded_columns.json"
)

save_dict_json(
    {
        "X_train_with_static_shape": list(X_train_trunc_ws_final.shape),
        "X_val_with_static_shape": list(X_val_trunc_ws_final.shape),
        "X_test_with_static_shape": list(X_test_trunc_ws_final.shape),
        "X_train_without_static_shape": list(X_train_trunc_ns_final.shape),
        "X_val_without_static_shape": list(X_val_trunc_ns_final.shape),
        "X_test_without_static_shape": list(X_test_trunc_ns_final.shape),
        "y_train_shape": list(y_train_trunc.shape),
        "y_val_shape": list(y_val_trunc.shape),
        "y_test_shape": list(y_test_trunc.shape),
        "n_static_encoded_columns": len(X_train_trunc_with_static.columns.tolist()),
    },
    TABULAR_DIR / "tabular_ablation_metadata.json"
)

Saved list to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\static_encoded_columns.json
Saved dictionary to: C:\Users\patri\OneDrive\Documents\SDSBO\Master Thesis\ssl-survival-scania-component-x\data\processed\tabular_truncated\tabular_ablation_metadata.json
